# Analiza streczeń literatury młodzieżowej z końca XX wieku

# Import bibliotek


In [ ]:
import nltk
import json
import morfeusz2
import string
import pandas as pd
import operator as op
import itertools as it
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [3]:
nltk.pathsec.ENFORCE = False
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\niksv\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# Utworzenie korpusu dokumentów

In [10]:
corpus_dir = "./literature"
corpus = PlaintextCorpusReader(corpus_dir, r".*\.txt")
files_names = corpus.fileids()

## Wstępne przygotowanie dokumentów

In [ ]:
documents_dict = {}
for file_name in files_names:
    documents_dict[file_name] = corpus.raw(file_name)
print(json.dumps(documents_dict, indent=4, ensure_ascii=False))


In [14]:
lengths = {}
for file_name in documents_dict:
    lengths[file_name] = {
        "pre": len(word_tokenize(documents_dict[file_name])),
    }
print(json.dumps(lengths, indent=4, ensure_ascii=False))

{
    "Harry Potter i Czara Ognia.txt": {
        "pre": 6748
    },
    "Harry Potter i Insygnia Smierci.txt": {
        "pre": 2997
    },
    "Harry Potter i Kamien Filozoficzny.txt": {
        "pre": 2236
    },
    "Harry Potter i Komnata Tajemnic.txt": {
        "pre": 1611
    },
    "Harry Potter i Ksiaze Polkrwi.txt": {
        "pre": 1993
    },
    "Harry Potter i Przeklete Dziecko.txt": {
        "pre": 3170
    },
    "Harry Potter i Wiezien Azkabanu.txt": {
        "pre": 2227
    },
    "Harry Potter i Zakon Feniksa.txt": {
        "pre": 1821
    },
    "Opowiesci z Narni. Kon i jego chlopiec.txt": {
        "pre": 179
    },
    "Opowiesci z Narni. Ksiaze Kaspian.txt": {
        "pre": 2376
    },
    "Opowiesci z Narni. Lew, czarownica i stara szafa.txt": {
        "pre": 2415
    },
    "Opowiesci z Narni. Ostatnia bitwa.txt": {
        "pre": 144
    },
    "Opowiesci z Narni. Podroz Wedrowca do Switu.txt": {
        "pre": 1155
    },
    "Opowiesci z Narni. Siostr

In [ ]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="utf-8")
stopwords = stopwords_file.read().splitlines()
stopwords = stopwords[4:]
stopwords_file.close()
stopwords


In [ ]:
def lematize(text):
    morfer = morfeusz2.Morfeusz()
    segments = it.groupby(morfer.analyse(text), op.itemgetter(0,1))
    def disambiguate(group):
        pairs = ((len(descr), lemma) for _ , _ , (_ , lemma, descr, _, _,) in group)
        perpl, lemma = min(pairs)
        return lemma.split(":")[0] if lemma else None
    lemmas = (disambiguate(group) for _, group in segments)
    return " ".join(filter(str.isalpha, lemmas))
    

In [ ]:
for file_name in documents_dict:
    lengths[file_name]["post"] = len(word_tokenize(documents_dict[file_name]))
print(json.dumps(lengths, indent=4, ensure_ascii=False))

In [4]:
morfer = morfeusz2.Morfeusz()
print(*morfer.analyse("Ala ma kota."), sep="\n")

(0, 1, ('Ala', 'Ala', 'subst:sg:nom:f', ['imię'], []))
(0, 1, ('Ala', 'Al', 'subst:sg:gen.acc:m1', ['imię'], []))
(0, 1, ('Ala', 'Alo', 'subst:sg:gen.acc:m1', ['imię'], []))
(1, 2, ('ma', 'mieć', 'fin:sg:ter:imperf', [], []))
(1, 2, ('ma', 'mój:A', 'adj:sg:nom.voc:f:pos', [], []))
(2, 3, ('kota', 'kota', 'subst:sg:nom:f', ['nazwa_pospolita'], []))
(2, 3, ('kota', 'kot:Sm1', 'subst:sg:gen.acc:m1', ['nazwa_pospolita'], ['pot.,środ.']))
(2, 3, ('kota', 'kot:Sm2', 'subst:sg:gen.acc:m2', ['nazwa_pospolita'], []))
(3, 4, ('.', '.', 'interp', [], []))


In [ ]:
lengths = pd.DataFrame.from_dict(lengths, orient="index")
lengths["change"] = lengths["post"] - lengths["pre"]
lengths["change_percent"] = (lengths["change"] / lengths["pre"] * 100).round(2)
lengths

## Utworzenie macierzy częstości

In [ ]:
documents_df = pd.DataFrame.from_dict(documents_dict, orient="index", columns=["text"])
documents_df

In [ ]:
matrix_tf = CountVectorizer().fit_transform(documents_df["content"])
matrix_tfidf = TfidfVectorizer().fit_transform(documents_df["content"])

In [ ]:
sparsity_tf = (1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100
sparsitry_tfidf = (1 - (matrix_tfidf.nnz / (matrix_tfidf.shape[0] * matrix_tfidf.shape[1]))) * 100